In [1]:

import time
from coffea import hist, nanoevents, util
import coffea.processor as processor
import awkward as ak
import numpy as np
import glob as glob
import re
import itertools
# import vector as vec
from coffea.nanoevents.methods import vector, candidate
from coffea.nanoevents import NanoAODSchema, BaseSchema
from coffea.lumi_tools import LumiMask
# for applying JECs
from coffea.lookup_tools import extractor
from coffea.jetmet_tools import CorrectedJetsFactory, JECStack, JetCorrectionUncertainty
from coffea.jetmet_tools import JetResolution, JetResolutionScaleFactor
# from jmeCorrections import JetMetCorrections
import json


import coffea.processor as processor
from coffea import hist
from coffea.analysis_tools import PackedSelection, Weights
from coffea.nanoevents import NanoAODSchema, NanoEventsFactory
from coffea.nanoevents.methods import nanoaod

# NanoAODSchema.warn_missing_crossrefs = False

import pickle
np.errstate(invalid='ignore', divide='ignore')


In [ ]:

def process(self, events):
        print("MonoTop")
        isData = not hasattr(events, "genWeight")

        if isData:
            # Nominal JEC are already applied in data
            return self.process_shift(events, None)

        import cachetools
        jec_cache = cachetools.Cache(np.inf)
#             nojer = "NOJER" if self._skipJER else ""
        jets = jet_factory[f"{self._year}mc{nojer}"].build(add_jec_variables(events.Jet, events.fixedGridRhoFastjetAll), jec_cache)
#         met = met_factory.build(events.MET, jets, {})

        shifts = [
            ({"Jet": jets, "MET": met}, None),
            ({"Jet": jets.JES_jes.up,  "MET": met.JES_jes.up}, "JESUp"),
            ({"Jet": jets.JES_jes.down, "MET": met.JES_jes.down}, "JESDown"),
            ({"Jet": jets, "MET": met.MET_UnclusteredEnergy.up}, "UESUp"),
            ({"Jet": jets, "MET": met.MET_UnclusteredEnergy.down}, "UESDown"),
        ]
        if not self._skipJER:
            shifts.extend([
                ({"Jet": jets.JER.up, "MET": met.JER.up}, "JERUp"),
                ({"Jet": jets.JER.down, "MET": met.JER.down}, "JERDown"),
            ])
        # HEM15/16 issue
        if self._year == "2018":
            _runid = (events.run >= 319077)
            j_mask = ak.where((jets.phi > -1.57) & (jets.phi < -0.87) &
                              (jets.eta > -2.5) & (jets.eta < 1.3) & _runid, 0.8, 1)
            fj_mask = ak.where((fatjets.phi > -1.57) & (fatjets.phi < -0.87) &
                               (fatjets.eta > -2.5) & (fatjets.eta < 1.3) & _runid, 
                               0.8, 1)
            shift_jets = copy.deepcopy(jets)
            shift_fatjets = copy.deepcopy(fatjets)
            for collection, mask in zip([shift_jets, shift_fatjets], [j_mask, fj_mask]):
                collection["pt"] = mask * collection.pt
                collection["mass"] = mask * collection.mass
            shifts.extend([
                ({"Jet": shift_jets, "MET": met}, "HEM18"),
            ])

        return processor.accumulate(self.process_shift(update(events, collections), name) for collections, name in shifts)